# 03 - Approve shared private link and upload NASA PDFs


## Step 1 - Approve pending Search->Storage private endpoint connection


In [ ]:
import os

from azure.identity import DefaultAzureCredential
from azure.mgmt.storage import StorageManagementClient
from azure.mgmt.storage.models import PrivateEndpointConnection, PrivateLinkServiceConnectionState
from dotenv import load_dotenv

load_dotenv("../env/.env")
subscription_id = os.getenv("AZURE_SUBSCRIPTION_ID")
resource_group = os.getenv("AZURE_RESOURCE_GROUP")
storage_account = os.getenv("STORAGE_ACCOUNT_NAME")
if not all([subscription_id, resource_group, storage_account]):
    raise RuntimeError("Missing AZURE_SUBSCRIPTION_ID, AZURE_RESOURCE_GROUP, or STORAGE_ACCOUNT_NAME in env/.env")

# The Search -> Storage shared private link surfaces as a pending PE connection
# on the storage account. The Azure CLI `az network private-endpoint-connection`
# command is a generic wrapper; the underlying REST call is on the storage RP,
# so we use StorageManagementClient.private_endpoint_connections directly.
credential = DefaultAzureCredential()
storage_client = StorageManagementClient(credential, subscription_id)
connections = list(storage_client.private_endpoint_connections.list(resource_group, storage_account))
pending = [c for c in connections if c.private_link_service_connection_state.status == "Pending"]
if not pending:
    print("No pending connection found. It may already be approved.")
else:
    for conn in pending:
        approval = PrivateEndpointConnection(
            private_link_service_connection_state=PrivateLinkServiceConnectionState(
                status="Approved",
                description="Approved for search01 demo",
            )
        )
        storage_client.private_endpoint_connections.put(
            resource_group_name=resource_group,
            account_name=storage_account,
            private_endpoint_connection_name=conn.name,
            properties=approval,
        )
        print(f"Approved: {conn.name}")


## Step 2 - Download NASA earth book PDFs


In [ ]:
from pathlib import Path
import requests

api_url = "https://api.github.com/repos/Azure-Samples/azure-search-sample-data/contents/nasa-e-book/earth_book_2019_text_pages"
target = Path("../data/nasa")
target.mkdir(parents=True, exist_ok=True)
items = requests.get(api_url, timeout=60).json()
pdf_items = [x for x in items if x.get("name", "").lower().endswith(".pdf")]
if not pdf_items:
    raise RuntimeError("No PDFs found in NASA sample data path.")
for item in pdf_items:
    content = requests.get(item["download_url"], timeout=120).content
    path = target / item["name"]
    path.write_bytes(content)
    print(f"Downloaded {path.name}")
print(f"Total files: {len(list(target.glob('*.pdf')))}")


## Step 3 - Upload PDFs to Blob with DefaultAzureCredential


In [ ]:
import os
from pathlib import Path

from azure.identity import DefaultAzureCredential
from azure.storage.blob import BlobServiceClient
from dotenv import load_dotenv

load_dotenv("../env/.env")
storage_account = os.getenv("STORAGE_ACCOUNT_NAME")
container_name = os.getenv("STORAGE_CONTAINER_NAME", "nasa")
if not storage_account:
    raise RuntimeError("STORAGE_ACCOUNT_NAME missing in env/.env")
account_url = f"https://{storage_account}.blob.core.windows.net"
credential = DefaultAzureCredential()
blob_service = BlobServiceClient(account_url=account_url, credential=credential)
container = blob_service.get_container_client(container_name)
files = sorted(Path("../data/nasa").glob("*.pdf"))
if not files:
    raise RuntimeError("No local PDF files found. Run Step 2 first.")
for file_path in files:
    with file_path.open("rb") as f:
        container.upload_blob(name=file_path.name, data=f, overwrite=True)
    print(f"Uploaded {file_path.name}")
print(f"Uploaded {len(files)} files")


---

Continue with `04_search.ipynb`.
